# Tier 3a (Claude) — Criterion-Level Reranker (Ceiling Estimate)

**Purpose:** Establish the upper bound for criterion-level reranking using Claude instead of a 7B logprob scorer.
If this beats Tier 2a (NDCG@10=0.6485), the criterion-level architecture is sound and worth distilling
into a fine-tuned open model (Stage B/C).

**Architecture:** clf-v4 → top-50 → per-criterion Claude assessment → label-score aggregation

For each (patient, criterion) pair Claude returns one of:
`included / not_included / excluded / not_excluded / not_enough_information`

Labels are mapped to scores and aggregated per trial for reranking.

**Requires:** `criteria_data.jsonl` on Drive (from `dataprep_criteria.ipynb`).

**Prior results (TREC22):**

| Stage | NDCG@10 |
|---|---|
| clf-v4 baseline | 0.6388 |
| Tier 2a: clf→Qwen flat-doc top-50 | 0.6485 |
| Tier 3a: clf→Qwen criterion-level top-50 | 0.6242 (worse than clf-v4) |
| Tier 3a (Claude): clf→Claude criterion-level top-50 | _this notebook_ |

In [ ]:
!pip install -q anthropic nest_asyncio
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets scikit-learn transformers tqdm
!pip install -q sympy==1.13.1

In [ ]:
import os
os.environ['HF_TOKEN']           = ''  # HuggingFace READ token
os.environ['ANTHROPIC_API_KEY']  = ''  # Anthropic API key (console.anthropic.com)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT       = '/content/drive/MyDrive/ct_data23'
QRELS_PATH      = f'{DATA_ROOT}/unified_qrels.jsonl'
CRITERIA_PATH   = f'{DATA_ROOT}/criteria_data.jsonl'
TIER2A_PATH     = f'{DATA_ROOT}/llm_reranker_results.json'
TIER3A_QWEN_PATH = f'{DATA_ROOT}/criteria_reranker_results.json'

CLF_CHECKPOINT  = 'semaj83/ctmatch-clf-v4'
CLAUDE_MODEL    = 'claude-sonnet-4-6'

CLF_GATE_ALL    = 0.7460
CLF_GATE_T22    = 0.6388
GATE_TOL        = 0.005

TOP_K_RERANK    = 50
MAX_CONCURRENT  = 15    # concurrent Claude API calls per topic
TREC22_ONLY     = True

In [ ]:
import json

topic2text   = {}
topic2source = {}
topic2rel    = {}   # topic_id -> {doc_id: label}

with open(QRELS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        tid = rec['topic_id']
        if TREC22_ONLY and rec['source'] != 'trec22':
            continue
        topic2text[tid] = rec['topic_text']
        topic2source[tid] = rec['source']
        if tid not in topic2rel:
            topic2rel[tid] = {}
        topic2rel[tid][rec['doc_id']] = rec['label']

print(f'Topics: {len(topic2text)}')
avg_pool = sum(len(d) for d in topic2rel.values()) / len(topic2rel)
print(f'Avg judged docs/topic: {avg_pool:.1f}')

In [ ]:
from datasets import load_dataset

index2docid_ds = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
doc_texts_ds   = load_dataset('semaj83/ctmatch_ir', data_files='doc_texts.txt',   split='train')

index2docid = [row['text'].strip() for row in index2docid_ds]
docid2index = {nct_id: idx for idx, nct_id in enumerate(index2docid)}
docid2text  = {nct_id: doc_texts_ds[idx]['text'] for idx, nct_id in enumerate(index2docid)}

all_judged = {nct_id for d in topic2rel.values() for nct_id in d}
print(f'Corpus: {len(index2docid):,}  |  Judged: {len(all_judged)}')

In [ ]:
nctid2crit = {}
with open(CRITERIA_PATH) as f:
    for line in f:
        rec = json.loads(line)
        nctid2crit[rec['nct_id']] = {
            'include_criteria': rec['include_criteria'],
            'exclude_criteria': rec['exclude_criteria'],
        }

judged_recs = [nctid2crit[n] for n in all_judged if n in nctid2crit]
inc_lens = [len(r['include_criteria']) for r in judged_recs]
exc_lens = [len(r['exclude_criteria']) for r in judged_recs]
print(f'Criteria loaded for {len(nctid2crit):,} trials')
print(f'Inc/trial — mean {sum(inc_lens)/len(inc_lens):.1f}  max {max(inc_lens)}')
print(f'Exc/trial — mean {sum(exc_lens)/len(exc_lens):.1f}  max {max(exc_lens)}')

In [ ]:
import math

def calc_ndcg(ranked_ids, doc2rel, k=10):
    dcg  = sum(doc2rel.get(d, 0) / math.log2(r + 1)
               for r, d in enumerate(ranked_ids[:k], start=1))
    idcg = sum(v / math.log2(r + 1)
               for r, v in enumerate(sorted(doc2rel.values(), reverse=True)[:k], start=1))
    return dcg / idcg if idcg > 0 else 0.0

def calc_mrr(ranked_ids, doc2rel):
    for rank, doc_id in enumerate(ranked_ids, start=1):
        if doc2rel.get(doc_id, 0) >= 2:
            return 1.0 / rank
    return 0.0

def eval_ranking(topic2ranked, topic2rel, k=10):
    ndcgs, mrrs = [], []
    for tid, ranked in topic2ranked.items():
        doc2rel = topic2rel.get(tid, {})
        ndcgs.append(calc_ndcg(ranked, doc2rel, k))
        mrrs.append(calc_mrr(ranked, doc2rel))
    return {
        f'ndcg@{k}': sum(ndcgs) / len(ndcgs) if ndcgs else 0.0,
        'mrr':       sum(mrrs)  / len(mrrs)  if mrrs  else 0.0,
        'n_topics':  len(ndcgs),
    }

## Stage 1 — clf-v4 gate
Reproduce clf-v4 ranking to build the top-50 candidate set per topic.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

clf_tokenizer = AutoTokenizer.from_pretrained(CLF_CHECKPOINT)
clf_model     = AutoModelForSequenceClassification.from_pretrained(CLF_CHECKPOINT)
clf_model.eval().cuda()

id2label     = clf_model.config.id2label
relevant_col = [int(k) for k, v in id2label.items() if v == 'relevant'][0]
print(f'Labels: {id2label}  |  relevant_col: {relevant_col}')

In [ ]:
from tqdm.auto import tqdm

def clf_batch_score(topic_text, doc_texts, batch_size=64):
    scores = []
    for i in range(0, len(doc_texts), batch_size):
        pairs = [(topic_text, dt) for dt in doc_texts[i:i+batch_size]]
        enc   = clf_tokenizer(pairs, padding=True, truncation=True,
                              max_length=512, return_tensors='pt').to(clf_model.device)
        with torch.no_grad():
            logits = clf_model(**enc).logits
        scores.extend(F.softmax(logits, dim=1)[:, relevant_col].cpu().tolist())
    return scores

clf_topic2ranked = {}
for tid in tqdm(topic2text, desc='clf scoring'):
    nct_ids = [nid for nid in topic2rel[tid] if nid in docid2text]
    texts   = [docid2text[nid] for nid in nct_ids]
    scores  = clf_batch_score(topic2text[tid], texts)
    clf_topic2ranked[tid] = sorted(zip(nct_ids, scores), key=lambda x: x[1], reverse=True)

In [ ]:
clf_res  = eval_ranking({tid: [n for n, _ in r] for tid, r in clf_topic2ranked.items()}, topic2rel)
clf_ndcg = clf_res['ndcg@10']
gate     = CLF_GATE_T22 if TREC22_ONLY else CLF_GATE_ALL
status   = '✓ PASS' if clf_ndcg >= gate - GATE_TOL else '✗ FAIL'
print(f'{status}  clf-v4  NDCG@10={clf_ndcg:.4f}  MRR={clf_res["mrr"]:.4f}  (gate={gate})')
if '✗' in status:
    raise RuntimeError('clf-v4 gate failed')

In [ ]:
import gc
del clf_model, clf_tokenizer
gc.collect()
torch.cuda.empty_cache()
print('clf-v4 freed — GPU no longer needed')

## Cost estimate
Count total criteria across clf-v4 top-50 for all topics before spending any money.

In [ ]:
n_api_calls = 0
for tid in topic2text:
    top_k = [nid for nid, _ in clf_topic2ranked[tid][:TOP_K_RERANK]]
    for nid in top_k:
        crit = nctid2crit.get(nid, {'include_criteria': [], 'exclude_criteria': []})
        n_api_calls += len(crit['include_criteria']) + len(crit['exclude_criteria'])

# Sonnet pricing: $3/1M input, $15/1M output
avg_in  = 450   # patient (~200) + criterion (~50) + system prompt (~200)
avg_out = 15    # just the label token(s)
cost_in  = n_api_calls * avg_in  / 1e6 * 3.0
cost_out = n_api_calls * avg_out / 1e6 * 15.0

print(f'Total criterion-level API calls : {n_api_calls:,}')
print(f'Estimated input tokens          : {n_api_calls * avg_in / 1e6:.1f}M')
print(f'Estimated cost (Sonnet)         : ${cost_in + cost_out:.2f}')
print()
print('Proceed to next cell to start scoring.')

## Stage 2 — criterion-level Claude scoring

Each (patient, criterion) pair is sent to Claude as an async API call.
Within each topic, all criteria across the top-50 trials are scored concurrently
(bounded by `MAX_CONCURRENT` semaphore). Topics are processed sequentially.

In [ ]:
import anthropic
import asyncio
import nest_asyncio
nest_asyncio.apply()   # allows asyncio.run() inside Jupyter/Colab

aclient = anthropic.AsyncAnthropic(api_key=os.environ['ANTHROPIC_API_KEY'])

SYSTEM_PROMPT = """You are a clinical trial eligibility assessor.
Given a patient description and a single eligibility criterion, determine whether the criterion applies.

Respond with exactly one label and nothing else:
- included          (patient meets this inclusion criterion)
- not_included      (patient does not meet this inclusion criterion)
- excluded          (this exclusion criterion applies to the patient)
- not_excluded      (this exclusion criterion does not apply to the patient)
- not_enough_information  (cannot determine from the patient description)"""

VALID_LABELS = {'included', 'not_included', 'excluded', 'not_excluded', 'not_enough_information'}

def make_criterion_prompt(patient_text: str, criterion: str, crit_type: str) -> str:
    return (
        f'Patient: {patient_text}\n\n'
        f'{crit_type.capitalize()} criterion: {criterion}\n\n'
        'Assess this criterion.'
    )

def parse_label(text: str) -> str:
    cleaned = text.strip().lower().replace(' ', '_').strip('.')
    # Exact match first, then longest-substring-first to avoid 'included' matching 'not_included'
    if cleaned in VALID_LABELS:
        return cleaned
    for label in sorted(VALID_LABELS, key=len, reverse=True):
        if label in cleaned:
            return label
    return 'not_enough_information'

print(f'Claude client ready  |  model: {CLAUDE_MODEL}')

In [ ]:
# Quick sanity check before running the full pipeline
sample_topic = list(topic2text.keys())[0]
sample_text  = topic2text[sample_topic]

async def _test():
    msg = await aclient.messages.create(
        model=CLAUDE_MODEL, max_tokens=30, system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': make_criterion_prompt(
            sample_text, 'Patient must have a diagnosis of cancer', 'inclusion'
        )}],
    )
    return msg.content[0].text

raw = asyncio.run(_test())
print(f'Raw response : {repr(raw)}')
print(f'Parsed label : {parse_label(raw)}')

In [ ]:
# Inclusion label → positive score when criterion is met
# Exclusion label → negative score when criterion fires
LABEL_SCORE = {
    'included':               +1.0,
    'not_included':           -1.0,
    'excluded':               -2.0,   # stronger penalty: active exclusion is decisive
    'not_excluded':           +1.0,
    'not_enough_information':  0.0,
}

def trial_score(inc_labels, exc_labels, agg):
    inc_s = [LABEL_SCORE[l] for l in inc_labels]
    exc_s = [LABEL_SCORE[l] for l in exc_labels]
    all_s = inc_s + exc_s
    if not all_s:
        return 0.0
    if agg == 'sum':
        return sum(all_s)
    if agg == 'mean':
        return sum(all_s) / len(all_s)
    if agg == 'strict_exc':
        # Any active exclusion → large penalty regardless of inclusion score
        if 'excluded' in exc_labels:
            return -10.0 + (sum(inc_s) / len(inc_s) if inc_s else 0.0)
        return sum(all_s)
    return 0.0

AGG_NAMES = ['sum', 'mean', 'strict_exc']

In [ ]:
from collections import defaultdict

async def score_one(patient_text, criterion, crit_type, semaphore, retries=3):
    prompt = make_criterion_prompt(patient_text, criterion, crit_type)
    async with semaphore:
        for attempt in range(retries):
            try:
                msg = await aclient.messages.create(
                    model=CLAUDE_MODEL, max_tokens=30, system=SYSTEM_PROMPT,
                    messages=[{'role': 'user', 'content': prompt}],
                )
                return parse_label(msg.content[0].text)
            except Exception as e:
                if attempt == retries - 1:
                    return 'not_enough_information'
                await asyncio.sleep(2 ** attempt)


async def score_topic(patient_text, nct_ids):
    """Score all criteria for all trials in nct_ids concurrently."""
    semaphore = asyncio.Semaphore(MAX_CONCURRENT)
    tasks, meta = [], []
    for nid in nct_ids:
        crit = nctid2crit.get(nid, {'include_criteria': [], 'exclude_criteria': []})
        for ct in crit['include_criteria']:
            tasks.append(score_one(patient_text, ct, 'inclusion', semaphore))
            meta.append((nid, 'inclusion'))
        for ct in crit['exclude_criteria']:
            tasks.append(score_one(patient_text, ct, 'exclusion', semaphore))
            meta.append((nid, 'exclusion'))
    labels = await asyncio.gather(*tasks)
    return list(zip(meta, labels))

In [ ]:
all_raw_labels = {}   # topic_id -> [(nct_id, crit_type, label), ...]
claude_topic2ranked = {agg: {} for agg in AGG_NAMES}
n_total_calls = 0

async def run_pipeline():
    global n_total_calls
    for tid in tqdm(topic2text, desc='Topics'):
        top_k = [nid for nid, _ in clf_topic2ranked[tid][:TOP_K_RERANK]]
        rest  = [nid for nid, _ in clf_topic2ranked[tid][TOP_K_RERANK:]]

        labeled = await score_topic(topic2text[tid], top_k)
        all_raw_labels[tid] = [(nid, ct, lbl) for (nid, ct), lbl in labeled]
        n_total_calls += len(labeled)

        # Accumulate labels per trial
        by_trial = defaultdict(lambda: {'inclusion': [], 'exclusion': []})
        for (nid, ct), lbl in labeled:
            by_trial[nid][ct].append(lbl)

        for agg in AGG_NAMES:
            scores  = {nid: trial_score(by_trial[nid]['inclusion'],
                                        by_trial[nid]['exclusion'], agg)
                       for nid in top_k}
            reranked = sorted(top_k, key=lambda n: scores[n], reverse=True)
            claude_topic2ranked[agg][tid] = reranked + rest

asyncio.run(run_pipeline())
print(f'Done. Total API calls: {n_total_calls:,}')

In [ ]:
# Label distribution across all criteria
from collections import Counter

all_labels = [lbl for rows in all_raw_labels.values() for _, _, lbl in rows]
dist = Counter(all_labels)
total = len(all_labels)
print('Label distribution:')
for label, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {label:30s} {count:6,}  ({100*count/total:.1f}%)')

In [ ]:
print(f'=== Claude criterion-level reranker (TREC22) ===')
print(f'  clf-v4 baseline:  NDCG@10={clf_ndcg:.4f}')
print()

agg_results = {}
for agg in AGG_NAMES:
    res = eval_ranking(claude_topic2ranked[agg], topic2rel)
    agg_results[agg] = res
    delta = res['ndcg@10'] - clf_ndcg
    sign  = '+' if delta >= 0 else ''
    print(f'  {agg:15s}  NDCG@10={res["ndcg@10"]:.4f}  MRR={res["mrr"]:.4f}  Δ={sign}{delta:.4f}')

In [ ]:
# Full cross-tier comparison
best_agg  = max(agg_results, key=lambda a: agg_results[a]['ndcg@10'])
best_ndcg = agg_results[best_agg]['ndcg@10']

def extract_ndcg(val):
    """Handle both plain-float and nested-dict result shapes."""
    if isinstance(val, dict):
        return float(val.get('ndcg@10', float('nan')))
    return float(val) if val is not None else float('nan')

rows = [('clf-v4', clf_ndcg)]
if os.path.exists(TIER2A_PATH):
    with open(TIER2A_PATH) as f:
        t2a = json.load(f)
    rows.append(('Tier 2a: clf→Qwen flat-doc', extract_ndcg(t2a.get('clf_llm_top50'))))
if os.path.exists(TIER3A_QWEN_PATH):
    with open(TIER3A_QWEN_PATH) as f:
        t3q = json.load(f)
    rows.append((f'Tier 3a: clf→Qwen criterion ({t3q.get("best_agg","?")})',
                 extract_ndcg(t3q.get('best_ndcg@10'))))
rows.append((f'Tier 3a: clf→Claude criterion ({best_agg})', best_ndcg))

print('\n--- Cross-tier comparison (TREC22 NDCG@10) ---')
for name, ndcg in rows:
    import math
    if math.isnan(ndcg):
        print(f'  N/A     (no data)  {name}')
    else:
        bar = '█' * int(ndcg * 80)
        print(f'  {ndcg:.4f}  {bar}  {name}')

nei_pct = 100 * sum(1 for rows in all_raw_labels.values()
                    for _, _, l in rows if l == 'not_enough_information') / max(n_total_calls, 1)
print(f'\nNEI rate: {nei_pct:.1f}% — patient descriptions lack information for {nei_pct:.0f}% of criteria')
print('Key finding: bottleneck is information content, not model quality.')

In [ ]:
output = {
    'clf_v4':              clf_ndcg,
    'agg_results':         agg_results,
    'best_agg':            best_agg,
    'best_ndcg@10':        best_ndcg,
    'best_mrr':            agg_results[best_agg]['mrr'],
    'n_api_calls':         n_total_calls,
    'model':               CLAUDE_MODEL,
    'top_k_rerank':        TOP_K_RERANK,
    'trec22_only':         TREC22_ONLY,
    'clf_checkpoint':      CLF_CHECKPOINT,
}

save_path = f'{DATA_ROOT}/criteria_claude_results.json'
with open(save_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Results  → {save_path}')

# Also save raw labels for future analysis / distillation
labels_path = f'{DATA_ROOT}/criteria_claude_labels.jsonl'
with open(labels_path, 'w') as f:
    for tid, rows in all_raw_labels.items():
        json.dump({'topic_id': tid, 'labels': [(nid, ct, lbl) for nid, ct, lbl in rows]}, f)
        f.write('\n')
print(f'Raw labels → {labels_path}  (use for distillation fine-tuning)')

print(f'\nBest: {best_agg}  NDCG@10={best_ndcg:.4f}  MRR={agg_results[best_agg]["mrr"]:.4f}')

## CoT prompt A/B test

Compare the simple-label prompt used above against a chain-of-thought prompt on 5 cases
chosen to exercise the main NEI failure modes.

| Case | Tests |
|---|---|
| Karnofsky ≥ 70 | Performance status — canonical NEI; CoT should infer from functional description |
| Ambulatory / self-care | ECOG-adjacent — can CoT infer from "ambulates independently"? |
| Psychiatric exclusion | Absence-of-condition reasoning — CoT may infer from no psychiatric history in topic |
| Calcium intake threshold | True ambiguity — expect NEI regardless of prompt |
| Prior corticosteroid use | Temporal + treatment history reasoning |

If CoT converts cases 1–2 from NEI to a label, the 41.8% NEI rate seen above is a
prompting artifact, not a fundamental information limit. That would justify running
the full pipeline again with the CoT prompt.

In [ ]:
SYSTEM_COT = """You are a clinical trial eligibility assessor with deep clinical expertise.
Given a patient description and a single eligibility criterion, assess whether the criterion applies.

Think through the assessment step by step:
1. What specific clinical information does this criterion require?
2. Is that information present explicitly in the patient description?
3. If not explicit, what can be reasonably inferred from the clinical context?
   - Apply standard clinical knowledge (e.g. "ambulates independently" → likely ECOG 0-1 or Karnofsky ≥ 70)
   - Expand abbreviations where confident (e.g. "AC chemo" → doxorubicin/cyclophosphamide)
   - Make calibrated inferences, noting your uncertainty
4. Assign a label based on your reasoning.

After your step-by-step reasoning, end with exactly one line in this format:
Label: [included / not_included / excluded / not_excluded / not_enough_information]"""

def parse_cot_label(text: str) -> str:
    for line in reversed(text.strip().split('\n')):
        if line.lower().startswith('label:'):
            return parse_label(line.split(':', 1)[1].strip())
    return parse_label(text)  # fallback: treat whole response as label

In [ ]:
import textwrap

# Pick the first 3 TREC22 topics as patient descriptions
_topic_items = list(topic2text.items())
P0, P1, P2 = _topic_items[0][1], _topic_items[1][1], _topic_items[2][1]

COT_TEST_CASES = [
    # (label, patient_text, crit_type, criterion)
    ('Karnofsky ≥ 70',
     P0, 'inclusion',
     'Patients must have a Karnofsky performance status great or equal to 70%'),

    ('Ambulatory / self-care',
     P1, 'inclusion',
     'Patient must be ambulatory and relatively good health. Even if unable to work '
     'at least able to partially care for self and not demented'),

    ('Psychiatric exclusion',
     P0, 'exclusion',
     'Major depression or another major psychiatric disorder as described in DSM IV '
     'within the past 2 years'),

    ('Calcium intake threshold',
     P2, 'inclusion',
     'Calcium intake below a threshold level'),

    ('Prior corticosteroid use',
     P1, 'exclusion',
     'Have used corticosteroids for more than 30 days within the past 90 days. '
     'Patients who have been off corticosteroids for at least 30 days may be eligible'),
]

async def _run_cot_test():
    results = []
    for label, patient_text, crit_type, criterion in COT_TEST_CASES:
        user_msg = make_criterion_prompt(patient_text, criterion, crit_type)
        simple_r, cot_r = await asyncio.gather(
            aclient.messages.create(model=CLAUDE_MODEL, max_tokens=20,
                                    system=SYSTEM_PROMPT,
                                    messages=[{'role': 'user', 'content': user_msg}]),
            aclient.messages.create(model=CLAUDE_MODEL, max_tokens=600,
                                    system=SYSTEM_COT,
                                    messages=[{'role': 'user', 'content': user_msg}]),
        )
        simple_label = parse_label(simple_r.content[0].text)
        cot_raw      = cot_r.content[0].text
        cot_label    = parse_cot_label(cot_raw)
        # Flag truncation so we know if Label: line was actually present
        truncated    = 'label:' not in cot_raw.lower()
        results.append((label, crit_type, criterion, patient_text,
                        simple_label, cot_label, cot_raw, truncated))
    return results

cot_results = asyncio.run(_run_cot_test())

In [ ]:
for label, crit_type, criterion, patient_text, simple_label, cot_label, cot_raw, truncated in cot_results:
    changed   = '  ← CHANGED' if simple_label != cot_label else ''
    trunc_tag = '  [TRUNCATED — Label: line missing]' if truncated else ''
    print(f'{"─"*72}')
    print(f'Case       : {label}  ({crit_type})')
    print(f'Criterion  : {textwrap.shorten(criterion, width=70)}')
    print(f'Patient    : {textwrap.shorten(patient_text, width=70)}')
    print(f'Simple     : {simple_label}')
    print(f'CoT        : {cot_label}{changed}{trunc_tag}')
    print(f'\nReasoning:')
    for line in cot_raw.strip().split('\n'):
        print(f'  {line}')
    print()

n_changed    = sum(1 for *_, s, c, __, ___ in cot_results if s != c)
n_truncated  = sum(1 for *_, t in cot_results if t)
n_nei_simple = sum(1 for *_, s, c, __, ___ in cot_results if s == 'not_enough_information')
n_nei_cot    = sum(1 for *_, s, c, __, ___ in cot_results if c == 'not_enough_information')
print(f'{"="*72}')
print(f'Truncated responses (Label: line missing) : {n_truncated}/{len(cot_results)}')
print(f'NEI rate — simple: {n_nei_simple}/{len(cot_results)}  |  CoT: {n_nei_cot}/{len(cot_results)}')
print(f'Labels changed by CoT: {n_changed}/{len(cot_results)}')